# HVAC Prediction Pipeline Notebook\n
\n
This notebook combines the functionality of the various Python scripts in the project (`split_dataset.py`, `core_components.py`, `utils.py`, `tcn_model.py`, `hybrid_model.py`, `train.py`, `predict_hvac.py`, `optimize.py`, `ensemble_train.py`, `run_pipeline.py`) into a single workflow.\n
\n
**Sections:**\n
1.  **Imports:** Load all necessary libraries.\n
2.  **Configuration:** Set parameters for data paths, model hyperparameters, training settings, etc. (Replaces command-line arguments).\n
3.  **Utility Functions:** Define helper functions (`add_time_features`, `safe_literal_eval`).\n
4.  **Core Components:** Define Dataset, Models (LSTM, TCN, Hybrid), and sequence creation logic.\n
5.  **Data Splitting:** Function to split the dataset into training and testing sets.\n
6.  **Training:** Functions for loading data, building models, training, evaluation, and artifact logging.\n
7.  **Prediction:** Functions for loading trained models and making predictions on new data.\n
8.  **Optimization (Optuna):** Functions for hyperparameter optimization using Optuna.\n
9.  **Ensemble Training:** Logic for training multiple models with different seeds.\n
10. **Pipeline Execution:** Example workflows for running training, prediction, optimization, or ensembling.

## 1. Imports

In [1]:
import os
import sys
import subprocess
import threading
import re
import argparse  # Keep for structure, but replace usage
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.parametrizations import weight_norm
import joblib
from tqdm.notebook import tqdm # Use notebook-friendly tqdm
import mlflow
import mlflow.pytorch
import random
import ast
import tempfile
import json
import optuna

# Ensure MLflow experiment is set (optional, good practice)
mlflow.set_experiment("HVAC_Prediction")

<Experiment: artifact_location='file:///home/ben/Documents/Projects/IFT3395-project/mlruns/888792303717812512', creation_time=1744968320304, experiment_id='888792303717812512', last_update_time=1744968320304, lifecycle_stage='active', name='HVAC_Prediction', tags={}>

## 2. Configuration
Define all parameters here. These replace the command-line arguments used in the original scripts.

In [2]:
# --- Configuration Object (Simulates argparse Namespace) ---
class Config:
    def __init__(self):
        # Data/Paths
        self.train_csv = 'dataset/train_split.csv'
        self.test_csv = 'dataset/test_split.csv' # Used in training evaluation
        self.predict_features_csv = 'dataset/test_features.csv' # Used for final prediction
        self.output_dir = 'models_lstm' # Base directory for saving models/scalers (LSTM specific)
        self.results_dir = 'results_lstm' # Base directory for saving results (LSTM specific)
        # Note: Filenames are often adapted dynamically within runs (training, ensemble, optuna)
        self.model_base_filename = 'lstm_hvac_best.pth'
        self.feature_scaler_base_filename = 'feature_scaler_lstm.save'
        self.target_scaler_base_filename = 'target_scaler_lstm.save'
        self.results_base_filename = 'test_set_eval_lstm.csv'
        self.submission_base_filename = 'submission_lstm.csv'

        # Model Hyperparameters (LSTM Only)
        self.seq_len = 16
        self.hidden_size = 128
        self.num_layers = 2
        self.dropout = 0.2

        # Training Hyperparameters
        self.batch_size = 64
        self.num_epochs = 50 # Max epochs
        self.lr = 1e-3
        self.loss_function = 'Huber' # 'MSE' or 'Huber'
        self.val_split_ratio = 0.15

        # Feature Engineering
        self.use_time_features = True

        # Training Enhancements
        self.use_lr_scheduler = True
        self.lr_scheduler_patience = 5
        self.lr_scheduler_factor = 0.5
        self.use_early_stopping = True
        self.early_stopping_patience = 10
        self.use_gradient_clipping = True
        self.gradient_clip_max_norm = 1.0

        # Prediction Specific
        self.training_run_id = None # SET THIS when running prediction based on a specific training run

        # Optimization Specific (Optuna)
        self.n_trials = 30
        self.study_name = "hvac_optimization_lstm_notebook"
        self.storage_url = None # e.g., "sqlite:///hvac_optuna_lstm_notebook.db"

        # Ensemble Specific
        self.random_seeds = [42, 123, 456, 789, 1011]
        self.ensemble_run_name = "hvac_ensemble_lstm_notebook"
        self.model_paths_filename = "ensemble_lstm_model_paths_notebook.json"

        # --- Dynamic names (set within functions like run_training_trial) ---
        # These are set based on trial/seed to avoid overwriting
        self.model_filename = self.model_base_filename
        self.feature_scaler_filename = self.feature_scaler_base_filename
        self.target_scaler_filename = self.target_scaler_base_filename
        self.results_filename = self.results_base_filename
        self.output_filename = self.submission_base_filename # For prediction output

    def __getitem__(self, key):
        # Allow dictionary-like access
        return getattr(self, key)

    def get(self, key, default=None):
        # Allow dictionary-like access with default
        return getattr(self, key, default)

    def __contains__(self, key):
        # Allow 'in' checks
        return hasattr(self, key)

# Instantiate the config
config = Config()

# Display config (optional)
import pprint
pprint.pprint(vars(config))

{'batch_size': 64,
 'dropout': 0.2,
 'early_stopping_patience': 10,
 'ensemble_run_name': 'hvac_ensemble_lstm_notebook',
 'feature_scaler_base_filename': 'feature_scaler_lstm.save',
 'feature_scaler_filename': 'feature_scaler_lstm.save',
 'gradient_clip_max_norm': 1.0,
 'hidden_size': 128,
 'loss_function': 'Huber',
 'lr': 0.001,
 'lr_scheduler_factor': 0.5,
 'lr_scheduler_patience': 5,
 'model_base_filename': 'lstm_hvac_best.pth',
 'model_filename': 'lstm_hvac_best.pth',
 'model_paths_filename': 'ensemble_lstm_model_paths_notebook.json',
 'n_trials': 30,
 'num_epochs': 50,
 'num_layers': 2,
 'output_dir': 'models_lstm',
 'output_filename': 'submission_lstm.csv',
 'predict_features_csv': 'dataset/test_features.csv',
 'random_seeds': [42, 123, 456, 789, 1011],
 'results_base_filename': 'test_set_eval_lstm.csv',
 'results_dir': 'results_lstm',
 'results_filename': 'test_set_eval_lstm.csv',
 'seq_len': 16,
 'storage_url': None,
 'study_name': 'hvac_optimization_lstm_notebook',
 'submissio

## 3. Utility Functions

In [3]:
# From utils.py

def add_time_features(df):
    """Adds time-based features to the DataFrame."""
    # Ensure 'date' column is datetime type
    if 'date' not in df.columns:
        raise ValueError("'date' column is required for time features but not found.")
    df['date'] = pd.to_datetime(df['date'])
    df['hour'] = df['date'].dt.hour
    df['dayofweek'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month
    # Optionally add sine/cosine transformations for cyclical features
    df['hour_sin'] = np.sin(2 * np.pi * df['hour']/24.0)
    df['hour_cos'] = np.cos(2 * np.pi * df['hour']/24.0)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12.0)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12.0)
    return df

# From utils.py (originally added in predict_hvac.py)
def safe_literal_eval(val):
    """Safely evaluate a string containing a Python literal (list, tuple, number, bool, string, etc.)."""
    try:
        # Handle boolean strings explicitly
        if isinstance(val, str):
            if val.lower() == 'true':
                return True
            if val.lower() == 'false':
                return False
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        # If it's not a literal, return the original string
        return val

## 4. Core Components

In [4]:
# From core_components.py

class HVACDataset(Dataset):
    def __init__(self, sequences, targets):
        self.X = sequences.astype(np.float32)
        # Ensure target is float32 and potentially reshape for loss function compatibility
        self.y = targets.astype(np.float32).reshape(-1, 1)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, dropout=0.0):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # Apply dropout between LSTM layers only if num_layers > 1
        lstm_dropout = dropout if num_layers > 1 else 0.0

        self.lstm = nn.LSTM(
            input_size,
            hidden_size,
            num_layers,
            batch_first=True,
            dropout=lstm_dropout
        )

        # Batch normalization after LSTM output (applied to hidden_size features)
        self.bn = nn.BatchNorm1d(hidden_size)

        # Dropout layer before final FC layer
        self.dropout = nn.Dropout(dropout)

        # Final fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, seq_len, input_size)
        out, (hn, cn) = self.lstm(x)

        # Use the output of the last time step
        # out shape: (batch, seq_len, hidden_size)
        # We want the features from the last time step: (batch, hidden_size)
        last_time_step_out = out[:, -1, :]

        # Apply batch normalization
        # BatchNorm1d expects input of shape (N, C) or (N, C, L)
        # Here N=batch_size, C=hidden_size
        bn_out = self.bn(last_time_step_out)

        # Apply dropout
        dropout_out = self.dropout(bn_out)

        # Final prediction
        final_out = self.fc(dropout_out)

        return final_out

def create_sequences(data, seq_len=16):
    """
    Create input sequences and targets for time series forecasting.
    data: tuple (features, consumption),
          features: np.array shape (N, n_features)
          consumption: np.array shape (N,)
    Returns:
      X: np.array shape (num_samples, seq_len, n_features)
      y: np.array shape (num_samples, 1) - Target is consumption at step t + seq_len + 15
    Target is a single consumption value 16 time steps after the *end* of the input sequence.
    """
    features, consumption = data
    X, y = [], []
    N = len(consumption)
    # Adjust loop range to ensure target index i + seq_len + 15 is valid
    for i in range(N - seq_len - 15):
        X.append(features[i:i+seq_len])
        # Target is the single value at step i + seq_len + 15
        y.append(consumption[i+seq_len+15])
    # Return y as a column vector
    return np.array(X), np.array(y)

# TCNModel, TemporalBlock, HybridModel REMOVED

## 5. Data Splitting

In [5]:
# From split_dataset.py

def split_data(input_csv=os.path.join('dataset', 'train_dataset.csv'), 
                 train_out=os.path.join('dataset', 'train_split.csv'), 
                 test_out=os.path.join('dataset', 'test_split.csv'), 
                 split_ratio=0.8):
    """Splits the dataset based on time."""
    print(f"Splitting {input_csv}...")
    try:
        df = pd.read_csv(input_csv, parse_dates=['date'])
    except FileNotFoundError:
        print(f"Error: Input file not found at {input_csv}")
        return
        
    df = df.sort_values('date').reset_index(drop=True)
    
    # Compute split index
    split_idx = int(len(df) * split_ratio)
    train_df = df.iloc[:split_idx]
    test_df = df.iloc[split_idx:]
    
    # Create output directory if it doesn't exist
    os.makedirs(os.path.dirname(train_out), exist_ok=True)
    os.makedirs(os.path.dirname(test_out), exist_ok=True)
    
    # Save files
    train_df.to_csv(train_out, index=False)
    test_df.to_csv(test_out, index=False)
    
    print(f'Saved training split ({len(train_df)} rows) to {train_out}')
    print(f'Saved test split ({len(test_df)} rows) to {test_out}')

# Example Usage (Uncomment to run)
# split_data()

## 6. Training

In [6]:
# --- Functions from train.py (adapted for notebook) ---

def load_series(csv_path, feature_cols, target_col, use_time_features=True):
    df = pd.read_csv(csv_path, parse_dates=['date'])
    df = df.sort_values('date').reset_index(drop=True)

    # Add time features before selecting columns
    if use_time_features:
        df = add_time_features(df) # Uses utility function
        # Update feature_cols to include new time features
        time_feature_cols = ['hour_sin', 'hour_cos', 'dayofweek', 'month_sin', 'month_cos']
        all_feature_cols = feature_cols + time_feature_cols
    else:
        all_feature_cols = feature_cols

    # Ensure all feature columns exist
    missing_cols = [col for col in all_feature_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"DataFrame missing required feature columns: {missing_cols}")

    features = df[all_feature_cols].values
    target = df[target_col].values
    return features, target, df, all_feature_cols # Return updated feature list

def load_and_prepare_data(args):
    # Note: 'args' is now the config object
    print("Loading and preparing data...")
    base_feature_cols = [
        'temperature_exterieure', 'humidite', 'ensoleillement',
        'vitesse_vent', 'direction_vent', 'temperature_interieure'
    ]
    target_col = 'puissance_cvac'

    # Load train data and get updated feature list
    feat_train_val, targ_train_val, df_train_val, all_feature_cols = load_series(
        args.train_csv, base_feature_cols, target_col, args.use_time_features
    )
    # Load test data using the same feature list
    feat_test, targ_test, df_test, _ = load_series(
        args.test_csv, base_feature_cols, target_col, args.use_time_features
    )
    print(f"Loaded {len(df_train_val)} train/val samples and {len(df_test)} test samples.")
    print(f"Using features: {all_feature_cols}")

    # --- Scale Features ---
    feat_scaler = MinMaxScaler()
    feat_train_val_s = feat_scaler.fit_transform(feat_train_val)
    feat_test_s = feat_scaler.transform(feat_test)

    # --- Scale Targets ---
    targ_scaler = MinMaxScaler()
    targ_train_val_s = targ_scaler.fit_transform(targ_train_val.reshape(-1,1)).flatten()
    targ_test_s = targ_scaler.transform(targ_test.reshape(-1,1)).flatten()

    # --- Create Sequences ---
    # Uses core_components function
    X_train_val, y_train_val = create_sequences((feat_train_val_s, targ_train_val_s), seq_len=args.seq_len)
    X_test, y_test = create_sequences((feat_test_s, targ_test_s), seq_len=args.seq_len)

    # --- Split Train/Validation ---
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=args.val_split_ratio, shuffle=False
    )
    print(f"Created {len(X_train)} training sequences, {len(X_val)} validation sequences, {len(X_test)} test sequences.")

    # --- DataLoaders ---
    # Uses core_components class
    train_ds = HVACDataset(X_train, y_train)
    val_ds = HVACDataset(X_val, y_val)
    test_ds = HVACDataset(X_test, y_test)

    # Consider adjusting num_workers based on your environment
    num_workers = 0 # Often safer for notebooks, adjust if needed
    train_loader = DataLoader(train_ds, batch_size=args.batch_size, shuffle=True, pin_memory=True, num_workers=num_workers)
    val_loader = DataLoader(val_ds, batch_size=args.batch_size, shuffle=False, pin_memory=True, num_workers=num_workers)
    test_loader = DataLoader(test_ds, batch_size=args.batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, feat_scaler, targ_scaler, all_feature_cols

def build_model(args, input_size, device):
    # Note: 'args' is now the config object
    # Simplified: Always build LSTMModel
    print(f"Initializing LSTM Model for {input_size} features...")
    # Uses core_components class
    model = LSTMModel(
        input_size=input_size,
        hidden_size=args.hidden_size,
        num_layers=args.num_layers,
        dropout=args.dropout
    )
    model.to(device)
    print("Model initialized and moved to device:", device)
    return model

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, args, device):
    # Note: 'args' is now the config object
    print("Starting training...")
    best_val_loss = float('inf')
    epochs_no_improve = 0
    # Use the dynamically set model_filename from args
    best_model_path = os.path.join(args.output_dir, args.model_filename)
    train_ds_len = len(train_loader.dataset) # Get length for loss calculation
    val_ds_len = len(val_loader.dataset) # Get length for loss calculation

    pbar_epochs = tqdm(range(1, args.num_epochs + 1), desc='Epochs')
    for epoch in pbar_epochs:
        # Training Phase
        model.train()
        train_loss = 0.0
        # Use notebook tqdm for inner loop
        train_pbar = tqdm(train_loader, desc=f'Train Epoch {epoch}', leave=False)
        for Xb, yb in train_pbar:
            Xb, yb = Xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimizer.zero_grad()
            out = model(Xb)
            loss = criterion(out, yb)
            loss.backward()
            if args.use_gradient_clipping:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=args.gradient_clip_max_norm)
            optimizer.step()
            train_loss += loss.item() * Xb.size(0)
            train_pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        avg_train_loss = train_loss / train_ds_len

        # Validation Phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for Xb, yb in val_loader:
                Xb, yb = Xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
                out_scaled = model(Xb)
                loss = criterion(out_scaled, yb)
                val_loss += loss.item() * Xb.size(0)

        avg_val_loss = val_loss / val_ds_len
        pbar_epochs.set_description(f'Epoch {epoch}/{args.num_epochs} - Train Loss: {avg_train_loss:.6f} - Val Loss: {avg_val_loss:.6f}')

        # LR Scheduling
        if args.use_lr_scheduler and scheduler:
            scheduler.step(avg_val_loss)

        # Early Stopping & Saving Best Model
        if avg_val_loss < best_val_loss:
            print(f'\nValidation loss decreased ({best_val_loss:.6f} --> {avg_val_loss:.6f}). Saving model to {best_model_path}...\n')
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), best_model_path)
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f'\nValidation loss did not improve for {epochs_no_improve} epoch(s).\n')
            if args.use_early_stopping and epochs_no_improve >= args.early_stopping_patience:
                print(f'\nEarly stopping triggered after {epoch} epochs.\n')
                break

        # Log metrics per epoch to MLflow
        mlflow.log_metric("train_loss", avg_train_loss, step=epoch)
        mlflow.log_metric("val_loss", avg_val_loss, step=epoch)
        # Log learning rate if scheduler is used
        if args.use_lr_scheduler and scheduler:
            current_lr = optimizer.param_groups[0]['lr']
            mlflow.log_metric("learning_rate", current_lr, step=epoch)

    print("Training finished.")
    # Check if the best model file actually exists before returning path
    if not os.path.exists(best_model_path):
        print(f"Warning: Best model path {best_model_path} was not created (training might have stopped early without improvement).")
        return None
    return best_model_path

def evaluate_model(model, best_model_path, test_loader, targ_scaler, device):
    # Note: 'best_model_path' can be None if training failed to save a model
    if best_model_path is None or not os.path.exists(best_model_path):
        print(f"Error: Best model path '{best_model_path}' not found or not provided. Cannot evaluate.")
        return None, None, None # Indicate failure

    print(f"Loading best model state from {best_model_path} for final evaluation...")
    try:
        model.load_state_dict(torch.load(best_model_path, map_location=device))
    except Exception as e:
        print(f"Error loading model state dict: {e}")
        print("Evaluation cannot proceed without a valid model state.")
        return None, None, None # Indicate failure

    model.eval()
    test_preds_scaled_list = []
    test_trues_scaled_list = []
    print("Running final evaluation on test set...")
    with torch.no_grad():
        for Xb, yb in tqdm(test_loader, desc='Evaluating', leave=False):
            Xb = Xb.to(device, non_blocking=True)
            # Model expects yb on device for potential internal use even in eval, though we use its numpy version
            yb_device = yb.to(device, non_blocking=True)
            out_scaled = model(Xb).cpu().numpy() # (batch, 1)
            test_preds_scaled_list.append(out_scaled)
            test_trues_scaled_list.append(yb.numpy()) # yb is already (batch, 1)

    preds_scaled = np.concatenate(test_preds_scaled_list) # (n_samples, 1)
    trues_scaled = np.concatenate(test_trues_scaled_list) # (n_samples, 1)

    # Inverse transform (scalers expect 2D array)
    preds = targ_scaler.inverse_transform(preds_scaled).flatten()
    trues = targ_scaler.inverse_transform(trues_scaled).flatten()

    # Calculate final metrics on test set
    mse = mean_squared_error(trues, preds)
    mae = mean_absolute_error(trues, preds)
    print(f'Final Test MSE: {mse:.4f}, MAE: {mae:.4f}')

    # Create results dataframe
    results_df = pd.DataFrame({'y_true': trues, 'y_pred': preds})

    return mse, mae, results_df

def log_artifacts(args, best_model_path, feat_scaler, targ_scaler, results_df, model, results_df_path=None):
    # Note: 'args' is now the config object
    print("Saving and logging artifacts...")

    output_dir = args.output_dir # Use the one defined in config/run
    results_dir = args.results_dir # Use the one defined in config/run

    # Ensure directories exist
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(results_dir, exist_ok=True)

    # --- Model ---
    # Model state dict already saved locally in train_model
    # Log the best model state dict found locally
    if best_model_path and os.path.exists(best_model_path):
        print(f"Logging best model state dict artifact from: {best_model_path}")
        mlflow.log_artifact(best_model_path, artifact_path="best_model")

        # --- Log Code for Reproducibility ---
        # Since we are in a notebook, logging individual script files is less relevant.
        # Instead, consider logging the notebook itself (if possible/desired)
        # Or log the key configuration parameters (already done via log_params)
        print("Skipping individual script logging in notebook environment.")
        # Optional: Log specific important code files if they exist outside the notebook
        # code_paths = [f for f in ["core_components.py", "lstm_model.py", ...] if os.path.exists(f)]
        # if code_paths:
        #     for code_path in code_paths:
        #          mlflow.log_artifact(code_path, artifact_path="code")
        #     print(f"Logged code artifacts: {code_paths}")
    else:
        print(f"Warning: Best model path '{best_model_path}' not found or not provided. Cannot log model artifact.")

    # --- Scalers ---
    # Use the dynamically set scaler filenames from args
    feature_scaler_path = os.path.join(output_dir, args.feature_scaler_filename)
    target_scaler_path = os.path.join(output_dir, args.target_scaler_filename)
    try:
        joblib.dump(feat_scaler, feature_scaler_path)
        joblib.dump(targ_scaler, target_scaler_path)
        print(f'Saved feature scaler to {feature_scaler_path}')
        print(f'Saved target scaler to {target_scaler_path}')
        mlflow.log_artifact(feature_scaler_path, artifact_path="scalers")
        mlflow.log_artifact(target_scaler_path, artifact_path="scalers")
        print("Logged scalers to MLflow artifacts path 'scalers'")
    except Exception as e:
         print(f"Error saving or logging scalers: {e}")

    # --- Evaluation Results ---
    if results_df is not None:
        # Use the provided path or construct from dynamic results_filename in args
        if results_df_path is None:
            results_df_path = os.path.join(results_dir, args.results_filename)
        try:
            results_df.to_csv(results_df_path, index=False)
            print(f'Saved test set evaluation results to {results_df_path}')
            mlflow.log_artifact(results_df_path, artifact_path="evaluation")
            print("Logged test evaluation CSV to MLflow artifacts path 'evaluation'")
        except Exception as e:
             print(f"Error saving or logging evaluation results: {e}")
    else:
        print("Skipping evaluation results logging as results_df is None.")

    # --- Hybrid Weights (REMOVED) ---
    print("Skipping hybrid weight logging (LSTM-only model).")

# --- Refactored Training and Evaluation Logic (Core Loop) ---
def run_training_trial(args, device, trial_num=None, random_seed=None):
    """
    Runs a single training and evaluation trial.
    Accepts args (config object or dict-like) and device.
    Optionally accepts an Optuna trial number and a random seed.
    Returns the metric to optimize (e.g., final test MAE) and the best model path.
    """
    # --- Set Seed ---
    current_seed = random_seed if random_seed is not None else random.randint(1, 10000)
    print(f"Setting random seed to: {current_seed}")
    torch.manual_seed(current_seed)
    np.random.seed(current_seed)
    random.seed(current_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(current_seed)
        # Potentially add deterministic options, but be aware of performance impact
        # torch.backends.cudnn.deterministic = True
        # torch.backends.cudnn.benchmark = False

    # Determine run name based on context (Optuna trial or seeded run)
    run_name_parts = []
    if trial_num is not None:
        run_name_parts.append(f"lstm_trial_{trial_num}")
    if random_seed is not None: # Distinguish explicitly seeded runs
        run_name_parts.append(f"lstm_seed_{random_seed}")
    elif trial_num is None: # Standalone run without explicit seed uses generated seed
         run_name_parts.append(f"lstm_run_seed_{current_seed}")

    run_name = "_".join(run_name_parts) if run_name_parts else "lstm_standalone_run"

    # Use nested run if part of Optuna or Ensemble
    is_nested = (trial_num is not None) or (random_seed is not None and 'ensemble_run_name' in args)

    with mlflow.start_run(run_name=run_name, nested=is_nested) as run:
        run_id = run.info.run_id
        print(f"MLflow Run ID: {run_id}") # Print for potential use by pipeline
        mlflow.log_param("run_type", run_name)
        mlflow.log_param("effective_seed", current_seed)
        mlflow.log_param("model_type", "LSTM") # Explicitly log model type

        run_context = "Standalone LSTM"
        if trial_num is not None and random_seed is not None:
            run_context = f"LSTM Optuna Trial {trial_num}, Seed {random_seed}"
        elif trial_num is not None:
            run_context = f"LSTM Optuna Trial {trial_num}"
        elif random_seed is not None:
            run_context = f"LSTM Ensemble Seed {random_seed}"

        print(f"--- Starting Run: {run_context} (MLflow Run ID: {run_id}) ---")
        # print("Args:", vars(args) if isinstance(args, Config) else args)

        # --- Adapt filenames for trial/seed to avoid overwrites ---
        output_dir = args.output_dir
        results_dir = args.results_dir
        model_base_filename = args.model_base_filename
        feature_scaler_base_filename = args.feature_scaler_base_filename
        target_scaler_base_filename = args.target_scaler_base_filename
        results_base_filename = args.results_base_filename

        # Add trial number and seed to filenames if provided
        trial_suffix = f"_trial_{trial_num}" if trial_num is not None else ""
        # Use the original random_seed if provided for ensemble, otherwise use current_seed
        seed_suffix = f"_seed_{random_seed if random_seed is not None else current_seed}"
        combined_suffix = f"{trial_suffix}{seed_suffix}"

        os.makedirs(output_dir, exist_ok=True)
        os.makedirs(results_dir, exist_ok=True)

        # Update args with trial/seed-specific paths/names for THIS run
        # We modify a copy or ensure args are treated as run-specific if needed
        # In this setup, run_training_trial receives 'args' which might be modified
        # If args is shared (like the global 'config'), create a copy
        run_args = argparse.Namespace(**vars(args)) # Create a copy
        run_args.model_filename = f"{os.path.splitext(model_base_filename)[0]}{combined_suffix}.pth"
        run_args.feature_scaler_filename = f"{os.path.splitext(feature_scaler_base_filename)[0]}{combined_suffix}.save"
        run_args.target_scaler_filename = f"{os.path.splitext(target_scaler_base_filename)[0]}{combined_suffix}.save"
        run_args.results_filename = f"{os.path.splitext(results_base_filename)[0]}{combined_suffix}.csv"
        # Also add trial_num to args if it's an optuna run, for logging
        if trial_num is not None:
             run_args.trial_num = trial_num

        # --- Log Parameters ---
        print("Logging parameters to MLflow...")
        params_to_log = vars(run_args)
        # Avoid logging potentially large/complex objects if not needed
        params_to_log.pop('random_seeds', None) # Logged separately in ensemble
        mlflow.log_params(params_to_log)
        mlflow.log_param("device", str(device))

        # Load data using the new function
        try:
            train_loader, val_loader, test_loader, feat_scaler, targ_scaler, all_feature_cols = load_and_prepare_data(run_args)
            mlflow.log_param("features", str(all_feature_cols)) # Log features here
        except Exception as e:
            print(f"Error during data loading: {e}")
            mlflow.set_tag("status", "failed_data_loading")
            mlflow.log_param("error", str(e))
            return float('inf'), None # Return high value for minimization if data fails

        # --- Model, Loss, Optimizer, Scheduler ---
        try:
            model = build_model(run_args, len(all_feature_cols), device) # Use the simplified build_model

            if run_args.loss_function == 'MSE':
                criterion = nn.MSELoss()
                print("Using MSELoss")
            elif run_args.loss_function == 'Huber':
                criterion = nn.HuberLoss()
                print("Using HuberLoss")
            else:
                raise ValueError(f"Unsupported loss function: {run_args.loss_function}")

            optimizer = torch.optim.Adam(model.parameters(), lr=run_args.lr)

            scheduler = None
            if run_args.use_lr_scheduler:
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer, mode='min',
                    factor=run_args.lr_scheduler_factor,
                    patience=run_args.lr_scheduler_patience,
                    verbose=True
                )
                print(f"Using ReduceLROnPlateau scheduler (patience={run_args.lr_scheduler_patience}, factor={run_args.lr_scheduler_factor})")
        except Exception as e:
            print(f"Error during model/optimizer setup: {e}")
            mlflow.set_tag("status", "failed_model_setup")
            mlflow.log_param("error", str(e))
            return float('inf'), None

        # --- Training Loop ---
        best_model_path = None # Initialize
        try:
            # Pass run_args to training function
            best_model_path = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, run_args, device)
        except Exception as e:
            print(f"Error during training: {e}")
            mlflow.set_tag("status", "failed_training")
            mlflow.log_param("error", str(e))
            # Optionally log the exception details
            # mlflow.log_text(str(e), "training_error.txt")
            return float('inf'), None # Indicate failure

        # --- Final Model Evaluation on Test Set ---
        test_mse, test_mae, results_df = None, None, None
        optimization_metric = float('inf') # Default to failure

        if best_model_path:
            try:
                test_mse, test_mae, results_df = evaluate_model(model, best_model_path, test_loader, targ_scaler, device)

                if test_mse is None: # Check if evaluation failed internally
                    print("Skipping artifact logging due to evaluation failure.")
                    mlflow.set_tag("status", "failed_evaluation")
                    optimization_metric = float('inf') # Return high value for minimization
                else:
                    # Log metrics before logging artifacts
                    mlflow.log_metric("final_test_mse", test_mse)
                    mlflow.log_metric("final_test_mae", test_mae)

                    # --- Saving/Logging Artifacts ---
                    # Adapt results_df path using the specific name
                    results_df_path = os.path.join(results_dir, run_args.results_filename)
                    log_artifacts(run_args, best_model_path, feat_scaler, targ_scaler, results_df, model, results_df_path=results_df_path)

                    optimization_metric = test_mae # Use MAE for optimization
                    mlflow.set_tag("status", "completed")

            except Exception as e:
                print(f"Error during evaluation or artifact logging: {e}")
                mlflow.set_tag("status", "failed_evaluation_logging")
                mlflow.log_param("error", str(e))
                optimization_metric = float('inf')
        else:
            print("Skipping evaluation and artifact logging because no best model was saved.")
            mlflow.set_tag("status", "failed_training_no_model")
            optimization_metric = float('inf')

        print(f"--- Finished Run: {run_context} | Test MAE: {optimization_metric if optimization_metric != float('inf') else 'N/A'} ---")
        # Return the metric Optuna should minimize (or final metric for single runs)
        # Also return the path to the saved model for ensemble purposes
        # Use the path determined within this run
        final_model_path = best_model_path if optimization_metric != float('inf') else None
        return optimization_metric, final_model_path


## 7. Prediction

In [7]:
# --- Functions from predict_hvac.py (adapted for notebook) ---

def load_training_info(client, training_run_id):
    print(f"Loading parameters and artifact URI from training run: {training_run_id}")
    try:
        training_run = client.get_run(training_run_id)
        train_params = training_run.data.params
        # train_artifact_uri = training_run.info.artifact_uri # Not strictly needed if using download_artifacts
        print(f"Successfully loaded info for run: {training_run_id}")
        return train_params # Return only params
    except Exception as e:
        raise ValueError(f"Could not fetch training run '{training_run_id}' from MLflow. Error: {e}")

def extract_training_params(train_params):
    print("Extracting and parsing parameters logged during training (LSTM)...")
    parsed_params = {}
    try:
        # Data related
        parsed_params['seq_len'] = int(train_params['seq_len'])
        parsed_params['use_time_features'] = safe_literal_eval(train_params.get('use_time_features', 'True')) # Use utility func

        # LSTM related
        parsed_params['hidden_size'] = int(train_params['hidden_size'])
        parsed_params['num_layers'] = int(train_params['num_layers'])
        parsed_params['dropout'] = float(train_params['dropout'])

        # TCN related (REMOVED)

        # Extract feature columns used during training
        all_feature_cols_str = train_params.get('features') # It was logged as a string representation of a list
        if not all_feature_cols_str:
            raise ValueError("'features' parameter not found in the training run.")
        all_feature_cols = safe_literal_eval(all_feature_cols_str) # Use utility func
        if not isinstance(all_feature_cols, list):
             raise ValueError(f"Could not parse 'features' from training params: '{all_feature_cols_str}'")
        parsed_params['all_feature_cols'] = all_feature_cols
        parsed_params['input_size'] = len(all_feature_cols)

        print(f"  Using seq_len: {parsed_params['seq_len']}, use_time_features: {parsed_params['use_time_features']}")
        print(f"  LSTM: hidden={parsed_params['hidden_size']}, layers={parsed_params['num_layers']}, dropout={parsed_params['dropout']}")
        print(f"  Features ({parsed_params['input_size']}): {parsed_params['all_feature_cols']}")

        return parsed_params

    except KeyError as e:
        raise ValueError(f"Missing expected parameter '{e}' in training run parameters.")
    except (ValueError, TypeError, SyntaxError) as e:
         raise ValueError(f"Error parsing parameter from training run. Error: {e}")

def download_artifacts(client, training_run_id, tmpdir, artifact_paths):
    """Downloads artifacts specified in artifact_paths dictionary."""
    print(f"Downloading artifacts from run {training_run_id} to {tmpdir}")
    downloaded_paths = {}
    try:
        for key, artifact_path in artifact_paths.items():
            print(f"  Downloading {key}: {artifact_path}...")
            local_path = client.download_artifacts(training_run_id, artifact_path, tmpdir)
            downloaded_paths[key] = local_path
            print(f"    Downloaded to: {local_path}")
        return downloaded_paths

    except Exception as e:
        raise RuntimeError(f"Failed to download artifacts from run '{training_run_id}'. Check paths and permissions. Error: {e}")

def prepare_test_data(predict_config, train_params, feature_scaler_local_path):
    # predict_config holds paths like test_csv, train_params holds model structure info
    print("Loading and preparing test data for prediction...")
    test_csv_path = predict_config.predict_features_csv # Use the specific csv for prediction
    use_time_features = train_params['use_time_features']
    all_feature_cols = train_params['all_feature_cols']
    input_size = train_params['input_size']
    seq_len = train_params['seq_len']

    # --- Load Test Data ---
    try:
        parse_dates = ['date'] if use_time_features else None
        df_test = pd.read_csv(test_csv_path, parse_dates=parse_dates)
    except FileNotFoundError:
        print(f"Error: Prediction features file not found at {test_csv_path}")
        raise

    if 'ID' not in df_test.columns:
        raise ValueError("Prediction features CSV must contain an 'ID' column.")
    ids = df_test['ID'].values

    # --- Add Time Features (if applicable) ---
    if use_time_features:
        print("Adding time features...")
        df_test = add_time_features(df_test) # Uses utility function
    else:
        print("Not using time features (as per training run).")

    # --- Validate and Select Features ---
    if not set(all_feature_cols).issubset(df_test.columns):
        missing_cols = set(all_feature_cols) - set(df_test.columns)
        raise ValueError(f'Prediction features CSV missing required feature columns used during training: {missing_cols}')
    features = df_test[all_feature_cols].values
    print(f"Using {input_size} features (determined from training run): {all_feature_cols}")

    # --- Load Scaler and Scale Features ---
    print(f"Loading feature scaler from: {feature_scaler_local_path}...")
    try:
        feat_scaler = joblib.load(feature_scaler_local_path)
    except FileNotFoundError:
         print(f"Error: Feature scaler file not found at {feature_scaler_local_path}")
         raise
    except Exception as e:
         print(f"Error loading feature scaler: {e}")
         raise

    num_features_expected_by_scaler = getattr(feat_scaler, 'n_features_in_', None)
    if num_features_expected_by_scaler is not None and input_size != num_features_expected_by_scaler:
         raise ValueError(
             f"Feature mismatch: Prepared {input_size} features based on training params, "
             f"but the loaded scaler expects {num_features_expected_by_scaler} features. "
             f"Inconsistency between training run parameters and saved scaler artifact."
         )

    print(f"Scaling {input_size} features...")
    features_scaled = feat_scaler.transform(features)

    # --- Input Sequence Preparation (Different from Training!) ---
    # For prediction, we typically need the *last* `seq_len` steps if predicting one step ahead.
    # However, the original predict script tiles the *current* features `seq_len` times.
    # We will replicate the original script's behavior here.
    # If the task requires actual lookback, this needs adjustment.
    print(f"Preparing input sequences using seq_len={seq_len} (tiling current features)...")
    num_samples = features_scaled.shape[0]
    num_features = features_scaled.shape[1]
    X_test_seq = np.zeros((num_samples, seq_len, num_features), dtype=np.float32)
    for i in range(num_samples):
        X_test_seq[i, :, :] = np.tile(features_scaled[i, :], (seq_len, 1))
    print(f"Prepared prediction input data shape: {X_test_seq.shape}")

    return X_test_seq, ids

def build_model_from_params(train_params):
    # Directly builds LSTMModel using extracted parameters
    print("Initializing LSTM model structure based on training parameters...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Directly build LSTMModel using extracted params
    model = LSTMModel(
        input_size=train_params['input_size'],
        hidden_size=train_params['hidden_size'],
        num_layers=train_params['num_layers'],
        dropout=train_params['dropout']
    )
    model.to(device)
    print("Model initialized and moved to device:", device)
    return model

def run_prediction(model, X_test_seq, target_scaler_local_path, device):
    print("Running predictions...")
    model.eval() # Ensure model is in eval mode
    preds_scaled_list = []

    # Create a simple DataLoader for prediction batches
    X_tensor = torch.from_numpy(X_test_seq).float()
    pred_dataset = torch.utils.data.TensorDataset(X_tensor)
    # Use a reasonable batch size, can be adjusted based on memory
    pred_loader = torch.utils.data.DataLoader(pred_dataset, batch_size=256, shuffle=False)

    with torch.no_grad():
         for Xb, in tqdm(pred_loader, desc="Predicting", leave=False):
            Xb = Xb.to(device)
            batch_preds_scaled = model(Xb).cpu().numpy() # Shape: (batch_size, 1)
            preds_scaled_list.append(batch_preds_scaled)

    preds_scaled = np.concatenate(preds_scaled_list) # Shape: (num_samples, 1)

    # --- Load Target Scaler and Inverse Transform ---
    print(f"Loading target scaler from: {target_scaler_local_path}...")
    try:
        target_scaler = joblib.load(target_scaler_local_path)
    except FileNotFoundError:
        print(f"Error: Target scaler file not found at {target_scaler_local_path}")
        raise
    except Exception as e:
        print(f"Error loading target scaler: {e}")
        raise

    preds = target_scaler.inverse_transform(preds_scaled).flatten() # Shape: (num_samples,)
    print(f"Generated {len(preds)} predictions.")
    return preds

def save_and_log_predictions(predict_config, ids, preds, prediction_run_info):
    # Uses predict_config for output paths/filenames
    print("Formatting predictions for submission...")
    results_dir = predict_config.results_dir
    output_filename = predict_config.output_filename # Use filename from config
    os.makedirs(results_dir, exist_ok=True)
    output_path = os.path.join(results_dir, output_filename)

    df_out = pd.DataFrame({
        'ID': ids,
        'puissance_cvac_future': preds
    })
    df_out.to_csv(output_path, index=False)
    print(f'Submission file saved locally to {output_path}')

    # --- Log Submission File to MLflow ---
    if prediction_run_info:
        print(f"Logging submission file '{output_filename}' to MLflow run {prediction_run_info.run_uuid}...")
        try:
            mlflow.log_artifact(output_path, artifact_path="predictions")
            print("Submission artifact logged.")
        except Exception as e:
            print(f"Error logging submission artifact: {e}")
    else:
        print("Skipping MLflow logging as prediction_run_info is not available.")
    return output_path


## 8. Optimization (Optuna)

In [8]:
# --- Functions from optimize.py (adapted for notebook) ---

def objective(trial, base_config):
    """Optuna objective function to train and evaluate the LSTM model."""
    # --- Device ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # --- Suggest Hyperparameters ---
    # Create a trial-specific config based on the base_config
    trial_args = argparse.Namespace(**vars(base_config)) # Copy base config

    # Override with suggested values for LSTM
    trial_args.lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    trial_args.num_layers = trial.suggest_int("num_layers", 1, 4)
    trial_args.batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    trial_args.hidden_size = trial.suggest_int("hidden_size", 32, 128, step=16)
    trial_args.dropout = trial.suggest_float("dropout", 0.0, 0.5) # Make dropout tunable

    # TCN parameters suggestion REMOVED

    # Add more parameters here if desired
    # trial_args.seq_len = trial.suggest_int("seq_len", 8, 32)
    # trial_args.loss_function = trial.suggest_categorical("loss_function", ['MSE', 'Huber'])
    # trial_args.early_stopping_patience = trial.suggest_int("early_stopping_patience", 5, 15)
    # trial_args.num_epochs = trial.suggest_int("num_epochs", 20, 100) # Make epochs tunable

    # --- Run the Training Trial ---
    # Pass the trial number for unique naming/logging within the trial's MLflow run
    # run_training_trial handles MLflow run creation (nested)
    # It returns (metric, model_path)
    test_mae, _ = run_training_trial(trial_args, device, trial_num=trial.number, random_seed=None)

    # --- Return Metric to Optimize ---
    # Optuna will minimize this value
    return test_mae

def run_optimization(config_obj):
    """Sets up and runs the Optuna study for LSTM."""
    # Create or load the study
    # Sampler: TPE is the default and usually works well.
    # Pruner: MedianPruner stops unpromising trials early.
    sampler = optuna.samplers.TPESampler()
    # Configure pruner - n_warmup_steps should relate to how many epochs it takes to get a reasonable loss
    # Adjust warmup steps based on typical convergence time or num_epochs used in optimization
    warmup_epochs = min(10, config_obj.num_epochs // 3) # Example heuristic
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=warmup_epochs, interval_steps=1)

    study = optuna.create_study(
        study_name=config_obj.study_name,
        storage=config_obj.storage_url,
        direction="minimize", # We want to minimize MAE
        sampler=sampler,
        pruner=pruner,
        load_if_exists=True # Set to True if using storage and want to resume
    )

    print(f"Starting optimization with {config_obj.n_trials} trials.")
    print(f"Study Name: {config_obj.study_name}")
    print(f"Sampler: {study.sampler.__class__.__name__}, Pruner: {study.pruner.__class__.__name__}")
    print(f"Using storage: {config_obj.storage_url or 'In-memory'}")

    # Start the optimization
    # Use a lambda to pass the base_config to the objective function
    try:
        study.optimize(lambda trial: objective(trial, config_obj),
                         n_trials=config_obj.n_trials,
                         timeout=None # Adjust timeout (seconds) if needed
                        )
    except KeyboardInterrupt:
        print("Optimization stopped manually.")

    # --- Results ---
    print("\nOptimization Finished!")
    print(f"Number of finished trials: {len(study.trials)}")

    try:
        best_trial = study.best_trial
        print(f"Best trial number: {best_trial.number}")
        print(f"  Value (MAE): {best_trial.value:.6f}")

        print("  Best Parameters:")
        for key, value in best_trial.params.items():
            print(f"    {key}: {value}")

        # Log best params to the parent optimization run if desired
        # Note: Optuna logs params per trial automatically within the trial's nested run
        # If you want them at the top level:
        # with mlflow.start_run(run_name=config_obj.study_name + "_parent", nested=False): # Example parent run
        #    mlflow.log_params(best_trial.params)
        #    mlflow.log_metric("best_mae", best_trial.value)

    except ValueError:
        print("No completed trials found in the study.")

    # Optional: Save results or visualize
    # try:
    #    df_results = study.trials_dataframe()
    #    results_path = os.path.join(config_obj.results_dir, f"{config_obj.study_name}_results.csv")
    #    df_results.to_csv(results_path, index=False)
    #    print(f"Optuna results saved to {results_path}")
    # except Exception as e:
    #    print(f"Could not save Optuna results dataframe: {e}")

    # Visualization requires plotly and kaleido
    # try:
    #     if optuna.visualization.is_available():
    #         fig1 = optuna.visualization.plot_optimization_history(study)
    #         fig1.show()
    #         fig2 = optuna.visualization.plot_param_importances(study)
    #         fig2.show()
    # except Exception as e:
    #     print(f"Could not generate Optuna plots: {e}")

    print("\nMLflow UI should show nested runs for each trial.")
    print(f"Access best trial's artifacts via MLflow run corresponding to trial {best_trial.number if 'best_trial' in locals() else 'N/A'}.")

    return study

## 9. Ensemble Training

In [10]:
# --- Functions from ensemble_train.py (adapted for notebook) ---

def run_ensemble_training(config_obj):
    """Runs training trials for multiple seeds defined in config_obj."""

    base_args = config_obj # Use the main config object as base
    random_seeds = config_obj.random_seeds
    ensemble_run_name = config_obj.ensemble_run_name
    model_paths_filename = config_obj.model_paths_filename

    print(f"Starting LSTM ensemble training with {len(random_seeds)} seeds: {random_seeds}")
    # print(f"Base arguments: {vars(base_args)}")

    # --- Determine Device ---
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # --- Parent MLflow Run for the Ensemble ---
    with mlflow.start_run(run_name=ensemble_run_name) as parent_run:
        parent_run_id = parent_run.info.run_id
        print(f"Parent Ensemble MLflow Run ID: {parent_run_id}")

        # Log base hyperparameters (excluding dynamic/seed-specific ones)
        parent_params = vars(base_args).copy()
        parent_params.pop('model_filename', None)
        parent_params.pop('feature_scaler_filename', None)
        parent_params.pop('target_scaler_filename', None)
        parent_params.pop('results_filename', None)
        parent_params.pop('training_run_id', None) # Not relevant here
        parent_params.pop('output_filename', None) # Prediction output filename
        mlflow.log_params(parent_params)
        mlflow.log_param("random_seeds_used", random_seeds)
        mlflow.log_param("num_ensemble_members_requested", len(random_seeds))
        mlflow.log_param("model_type", "LSTM")

        successful_model_paths = []
        successful_run_ids = []
        failed_seeds = []

        # --- Loop Through Seeds ---
        for seed in random_seeds:
            print(f"\n--- Training LSTM Ensemble Member: Seed {seed} ---")
            # We need to pass the base_args and the specific seed
            # run_training_trial will handle nested MLflow run creation
            try:
                # Add ensemble_run_name to args temporarily so run_training_trial knows it's nested
                args_copy = argparse.Namespace(**vars(base_args))
                args_copy.ensemble_run_name = ensemble_run_name # Mark as ensemble member

                # run_training_trial returns (metric, model_path)
                _, model_path = run_training_trial(args_copy, device, trial_num=None, random_seed=seed)

                # Get the run_id of the child run
                # Be careful, last_active_run might not be reliable in all execution environments
                # It's better if run_training_trial returned the run_id
                # For now, let's assume it works or retrieve later if needed
                child_run = mlflow.last_active_run()
                child_run_id = child_run.info.run_id if child_run else "Unknown"


                if model_path and os.path.exists(model_path):
                    print(f"Successfully trained model for seed {seed}. Path: {model_path}, Run ID: {child_run_id}")
                    # Store relative path for portability if possible
                    relative_path = os.path.relpath(model_path, start=os.getcwd()) # Or use absolute path
                    successful_model_paths.append(relative_path)
                    successful_run_ids.append(child_run_id)
                else:
                     print(f"Training failed or model path not returned for seed {seed}. Run ID: {child_run_id}")
                     failed_seeds.append(seed)

            except Exception as e:
                print(f"Error training model for seed {seed}: {e}")
                failed_seeds.append(seed)
                # Optionally log the error to the parent run tag
                mlflow.set_tag(f"seed_{seed}_error", str(e))
                # Decide if you want to stop the whole ensemble run on one failure
                # continue

        # --- Log Ensemble Results to Parent Run ---
        print("\n--- Ensemble Training Summary ---")
        print(f"Successfully trained models: {len(successful_model_paths)}")
        print(f"Failed seeds: {failed_seeds}")
        print("Successful model paths:")
        for path in successful_model_paths:
            print(f"  - {path}")
        print("Successful run IDs:")
        for run_id in successful_run_ids:
             print(f"  - {run_id}")

        # Log the list of successful model paths and run IDs to a file within the parent run artifacts
        output_dir = base_args.output_dir # Use output dir defined in config
        os.makedirs(output_dir, exist_ok=True)
        paths_filepath = os.path.join(output_dir, model_paths_filename)

        ensemble_info = {
            "successful_model_paths": successful_model_paths,
            "successful_run_ids": successful_run_ids,
            "failed_seeds": failed_seeds
        }

        try:
            with open(paths_filepath, 'w') as f:
                json.dump(ensemble_info, f, indent=4)
            print(f"Saved ensemble info to: {paths_filepath}")
            mlflow.log_artifact(paths_filepath, artifact_path="ensemble_info")
            print(f"Logged '{model_paths_filename}' to MLflow artifacts path 'ensemble_info'")
        except Exception as e:
            print(f"Error saving or logging ensemble info file: {e}")

        # Log overall status metrics and tags to parent run
        mlflow.log_metric("num_successful_members", len(successful_model_paths))
        mlflow.log_metric("num_failed_members", len(failed_seeds))
        if failed_seeds:
             mlflow.set_tag("status", "completed_with_failures")
        elif not successful_model_paths:
             mlflow.set_tag("status", "failed")
        else:
             mlflow.set_tag("status", "completed")

    return parent_run_id, ensemble_info

## 10. Pipeline Execution\n
\n
This section demonstrates how to run the different parts of the pipeline (training, prediction, optimization, ensemble) by calling the functions defined above.\n
\n
**Note:** The original `run_pipeline.py` used `subprocess` to execute scripts and captured stdout to find the MLflow Run ID. Here, we will call functions directly. The `run_training_trial` function now prints the Run ID, and we can capture it or retrieve it from MLflow if needed for subsequent steps like prediction.

In [11]:
# --- Top-Level Workflow Functions (Adapted from original Cell 10) ---

# Example: Run a Single Training Trial (LSTM Only)
def run_single_training(config_obj, seed=None):
    print("--- Running Single Training Trial (LSTM Only) ---")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")

    # Ensure we use a copy so modifications within the trial don't affect the original object
    run_args = argparse.Namespace(**vars(config_obj))

    # Call the core training function (already simplified for LSTM)
    final_metric, model_path = run_training_trial(run_args, device, trial_num=None, random_seed=seed)

    # Retrieve the run ID of the trial just executed
    last_run = mlflow.last_active_run()
    run_id = last_run.info.run_id if last_run else None
    print(f"Training Run ID: {run_id}")

    if final_metric == float('inf') or model_path is None:
        print("Standalone training run failed or did not produce a model.")
        return None, None
    else:
        print(f"Standalone training run finished. Final Test MAE: {final_metric:.6f}")
        print(f"Model saved to: {model_path}")
        return run_id, model_path

# Example: Run Prediction based on a Training Run ID (LSTM Only)
def run_prediction_pipeline(config_obj, training_run_id_to_use):
    print(f"--- Running Prediction Pipeline (LSTM Model) for Training Run ID: {training_run_id_to_use} ---")
    if not training_run_id_to_use:
        print("Error: Training Run ID must be provided for prediction.")
        return None

    # Create a prediction-specific config or modify a copy
    predict_config = argparse.Namespace(**vars(config_obj))
    predict_config.training_run_id = training_run_id_to_use
    # Set a specific output filename for this prediction run (can be overridden later)
    if not predict_config.output_filename or predict_config.output_filename == predict_config.submission_base_filename:
         predict_config.output_filename = f"submission_lstm_{training_run_id_to_use[:8]}.csv"

    # --- Start Prediction MLflow Run ---
    with mlflow.start_run(run_name=f"prediction_lstm_for_{training_run_id_to_use}") as prediction_run:
        prediction_run_info = prediction_run.info
        print(f"Started Prediction MLflow Run: {prediction_run_info.run_uuid}")
        mlflow.log_param("training_run_id", predict_config.training_run_id)
        mlflow.set_tag("mlflow.note.content", f"LSTM Predictions based on training run: {predict_config.training_run_id}")
        mlflow.log_param("model_type", "LSTM") # Explicitly log model type

        try:
            # --- Get Training Run Info ---
            client = mlflow.tracking.MlflowClient()
            train_params_raw = load_training_info(client, predict_config.training_run_id)

            # --- Extract Parameters from Training Run ---
            # Uses the simplified extract_training_params function
            train_params = extract_training_params(train_params_raw)

            # --- Determine Artifact Paths from Training Run ---
            # Attempt to reconstruct artifact filenames based on logged params/tags
            train_run_data = client.get_run(predict_config.training_run_id).data
            train_run_params = train_run_data.params
            train_run_tags = train_run_data.tags

            # Reconstruct suffix
            trial_num_str = train_run_params.get('trial_num')
            seed_str = train_run_params.get('effective_seed')

            trial_suffix = f"_trial_{trial_num_str}" if trial_num_str is not None and trial_num_str != 'None' else ""
            seed_suffix = f"_seed_{seed_str}" if seed_str is not None and seed_str != 'None' else ""

            # Handle potential case where run was standalone without explicit seed/trial in params
            if not seed_suffix and not trial_suffix:
                 # Maybe it was a single run where seed was passed directly?
                 # Or an older run? Check tags for seed info as fallback.
                 found_seed_in_tags = None
                 for tag_key, tag_value in train_run_tags.items():
                      # Look for tags like 'seed_42' or 'effective_seed' etc.
                      if 'seed' in tag_key.lower():
                           # Extract number if possible
                           match = re.search(r'\d+', tag_value)
                           if match:
                                found_seed_in_tags = match.group(0)
                                break
                           elif tag_value.isdigit(): # If tag value itself is the seed
                                found_seed_in_tags = tag_value
                                break
                 if found_seed_in_tags:
                      seed_suffix = f"_seed_{found_seed_in_tags}"
                 else:
                      print("Warning: Could not determine seed or trial number from training run parameters/tags. Assuming base artifact names without suffix.")

            combined_suffix = f"{trial_suffix}{seed_suffix}"

            # Use base names from config (make sure they are LSTM specific)
            model_artifact_filename = f"{os.path.splitext(predict_config.model_base_filename)[0]}{combined_suffix}.pth"
            feature_scaler_artifact_filename = f"{os.path.splitext(predict_config.feature_scaler_base_filename)[0]}{combined_suffix}.save"
            target_scaler_artifact_filename = f"{os.path.splitext(predict_config.target_scaler_base_filename)[0]}{combined_suffix}.save"

            artifact_paths_to_download = {
                'model': f"best_model/{model_artifact_filename}",
                'feature_scaler': f"scalers/{feature_scaler_artifact_filename}",
                'target_scaler': f"scalers/{target_scaler_artifact_filename}"
            }
            print(f"Attempting to download artifacts: {artifact_paths_to_download}")

            # --- Download Artifacts ---
            with tempfile.TemporaryDirectory() as tmpdir:
                try:
                     downloaded_paths = download_artifacts(client, predict_config.training_run_id, tmpdir, artifact_paths_to_download)
                     model_local_path = downloaded_paths['model']
                     feature_scaler_local_path = downloaded_paths['feature_scaler']
                     target_scaler_local_path = downloaded_paths['target_scaler']
                except Exception as download_error:
                     print(f"Error downloading artifacts: {download_error}")
                     print("Check if artifact paths/names match those logged in the training run.")
                     # You might need to manually inspect the training run artifacts in MLflow UI
                     # Example: mlflow.list_artifacts(predict_config.training_run_id)
                     raise # Re-raise the exception to stop the pipeline

                # --- Prepare Test Data (using predict_features_csv) ---
                # Uses the simplified prepare_test_data function
                X_test_seq, ids = prepare_test_data(predict_config, train_params, feature_scaler_local_path)

                # --- Load Model State ---
                print(f"Loading model state dict from downloaded artifact: {model_local_path}...")
                device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
                print(f"Using device: {device}")

                # Initialize model structure using parameters from training run
                # Uses the simplified build_model_from_params
                model = build_model_from_params(train_params)

                # Load the state dict
                model.load_state_dict(torch.load(model_local_path, map_location=device))
                model.to(device)
                print("Model state loaded successfully.")

                # --- Run Predictions ---
                # Uses the simplified run_prediction function
                preds = run_prediction(model, X_test_seq, target_scaler_local_path, device)

                # --- Format and Save/Log Submission ---
                # Uses the simplified save_and_log_predictions function
                submission_path = save_and_log_predictions(predict_config, ids, preds, prediction_run_info)
                print(f"Prediction script finished. Submission file: {submission_path}")
                mlflow.set_tag("status", "completed")
                return submission_path

        except Exception as e:
            print(f"Error during prediction pipeline: {e}")
            import traceback
            traceback.print_exc() # Print full traceback for debugging
            mlflow.set_tag("status", "failed_prediction")
            mlflow.log_param("error", str(e))
            return None

In [10]:
# 1. Define Configuration (use the 'config' object defined earlier or customize)
workflow_config = Config() # Start with defaults for LSTM
# workflow_config.num_epochs = 5 # Example: override for a quick test

# 2. Run Training
# This uses the simplified run_training_trial which only handles LSTM
training_run_id, saved_model_path = run_single_training(workflow_config, seed=42)

# 3. Run Prediction (if training was successful)
if training_run_id:
    # Pass the same config, but the function uses training_run_id to fetch specifics
    # This uses the simplified prediction pipeline
    submission_file = run_prediction_pipeline(workflow_config, training_run_id)
    if submission_file:
        print(f"\nWorkflow completed. Submission file at: {submission_file}")
else:
    print("\nTraining failed, skipping prediction.")

### Example Workflow: Train then Predict

In [ ]:
# 1. Define Configuration (use the 'config' object defined earlier or customize)
workflow_config = Config() # Start with defaults
# workflow_config.num_epochs = 5 # Example: override for a quick test
workflow_config.use_tcn = False # Ensure TCN is used if desired

# 2. Run Training
training_run_id, saved_model_path = run_single_training(workflow_config, seed=42) 

# 3. Run Prediction (if training was successful)
if training_run_id:
    # Pass the same config, but the function uses training_run_id to fetch specifics
    submission_file = run_prediction_pipeline(workflow_config, training_run_id)
    if submission_file:
        print(f"\nWorkflow completed. Submission file at: {submission_file}")
else:
    print("\nTraining failed, skipping prediction.")

### Example Workflow: Run Hyperparameter Optimization

In [12]:
# 1. Define Configuration for Optimization
opt_config = Config() # Start with defaults for LSTM
opt_config.n_trials = 30 # Number of trials to run
opt_config.num_epochs = 20 # Use fewer epochs for faster optimization trials
opt_config.study_name = "hvac_optimization_lstm_workflow"
# opt_config.storage_url = "sqlite:///hvac_optuna_lstm_workflow.db" # Optional: Use DB storage

# 2. Run Optimization
# This will create nested MLflow runs for each trial
print("--- Starting Hyperparameter Optimization ---")
# Uses the simplified objective function
study = run_optimization(opt_config)
print("--- Hyperparameter Optimization Finished ---")

# 3. Find the Best Trial's Run ID
best_run_id = None
try:
    if not study.trials:
            raise ValueError("Optimization study has no trials.")
    if study.best_trial is None:
            raise ValueError("Optimization study has no completed trials.")

    best_trial_num = study.best_trial.number
    print(f"Best Optuna trial number: {best_trial_num} with MAE: {study.best_value:.6f}")
    print("Finding corresponding MLflow run ID...")

    # Search MLflow for the run corresponding to the best trial
    client = mlflow.tracking.MlflowClient()

    # Try finding experiment by study name first
    experiment = client.get_experiment_by_name(opt_config.study_name)
    if not experiment:
        # Fallback: Try finding the experiment set at the beginning of the notebook
        experiment = client.get_experiment_by_name("HVAC_Prediction_LSTM_Only") # Updated experiment name

    if experiment:
        experiment_id = experiment.experiment_id
        print(f"Searching in experiment '{experiment.name}' (ID: {experiment_id})")
        # Note: This relies on the 'trial_num' parameter being logged correctly by run_training_trial
        runs = client.search_runs(
            experiment_ids=[experiment_id],
            filter_string=f"params.trial_num = '{best_trial_num}'", # Parameter is logged as string
            run_view_type=mlflow.entities.ViewType.ACTIVE_ONLY,
            max_results=1
        )
        if runs:
            best_run_id = runs[0].info.run_id
            print(f"Found MLflow run ID for best trial: {best_run_id}")
        else:
            # Attempt search without param filter if first fails (maybe param wasn't logged?)
            print(f"Warning: Could not find MLflow run for trial number {best_trial_num} using param filter. Searching by run name...")
            runs_by_name = client.search_runs(
                experiment_ids=[experiment_id],
                    # Construct expected run name: assumes format 'lstm_trial_{trial_num}'
                filter_string=f"tags.`mlflow.runName` = 'lstm_trial_{best_trial_num}'",
                run_view_type=mlflow.entities.ViewType.ACTIVE_ONLY,
                max_results=1
            )
            if runs_by_name:
                best_run_id = runs_by_name[0].info.run_id
                print(f"Found MLflow run ID by name: {best_run_id}")
            else:
                print(f"Warning: Could not find MLflow run for trial number {best_trial_num} by param or name in experiment '{experiment.name}'.")
    else:
        print("Warning: Could not find MLflow experiment. Cannot retrieve best run ID.")

except ValueError as e:
    print(f"Optimization Error: {e}")
except Exception as e:
    print(f"An error occurred while retrieving the best run ID: {e}")

# 4. Run Prediction using the Best Model from Optimization
if best_run_id:
    print(f"\n--- Running Prediction using Best Model from Optuna (Run ID: {best_run_id}) ---")
    # Use a fresh config object for prediction, run_prediction_pipeline fetches specifics
    pred_config = Config()
    # Create a unique output filename for this prediction run
    pred_config.output_filename = f"submission_lstm_optuna_best_{best_run_id[:8]}.csv" # Add run id snippet

    submission_file = run_prediction_pipeline(pred_config, best_run_id)
    if submission_file:
        print(f"\nOptimized Prediction Workflow completed. Submission file at: {submission_file}")
else:
    print("\nOptimization did not yield a best run ID, skipping prediction.")


[I 2025-04-18 20:33:21,087] A new study created in memory with name: hvac_optimization_lstm_workflow


--- Starting Hyperparameter Optimization ---
Starting optimization with 30 trials.
Study Name: hvac_optimization_lstm_workflow
Sampler: TPESampler, Pruner: MedianPruner
Using storage: In-memory
Setting random seed to: 5526
MLflow Run ID: 094b3607155d45b09d1a9893df354a3c
--- Starting Run: LSTM Optuna Trial 0 (MLflow Run ID: 094b3607155d45b09d1a9893df354a3c) ---
Logging parameters to MLflow...
Loading and preparing data...
Loaded 28032 train/val samples and 7008 test samples.
Using features: ['temperature_exterieure', 'humidite', 'ensoleillement', 'vitesse_vent', 'direction_vent', 'temperature_interieure', 'hour_sin', 'hour_cos', 'dayofweek', 'month_sin', 'month_cos']
Created 23800 training sequences, 4201 validation sequences, 6977 test sequences.
Initializing LSTM Model for 11 features...
Model initialized and moved to device: cpu
Using HuberLoss


/home/ben/Documents/Projects/IFT3395-project/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0


Using ReduceLROnPlateau scheduler (patience=5, factor=0.5)
Starting training...


/home/ben/Documents/Projects/IFT3395-project/.venv/lib/python3.13/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epochs:   0%|          | 0/20 [00:00<?, ?it/s]

... (1,258 outputs left out)


Validation loss did not improve for 10 epoch(s).


Early stopping triggered after 19 epochs.

Training finished.
Loading best model state from models_lstm/lstm_hvac_best_trial_29_seed_3612.pth for final evaluation...
Running final evaluation on test set...


Evaluating:   0%|          | 0/219 [00:00<?, ?it/s]

[I 2025-04-18 20:57:31,754] Trial 29 finished with value: 0.4439571797847748 and parameters: {'lr': 0.0009228940863175952, 'num_layers': 2, 'batch_size': 32, 'hidden_size': 80, 'dropout': 0.23528724520497635}. Best is trial 16 with value: 0.43219348788261414.


Final Test MSE: 0.3787, MAE: 0.4440
Saving and logging artifacts...
Logging best model state dict artifact from: models_lstm/lstm_hvac_best_trial_29_seed_3612.pth
Skipping individual script logging in notebook environment.
Saved feature scaler to models_lstm/feature_scaler_lstm_trial_29_seed_3612.save
Saved target scaler to models_lstm/target_scaler_lstm_trial_29_seed_3612.save
Logged scalers to MLflow artifacts path 'scalers'
Saved test set evaluation results to results_lstm/test_set_eval_lstm_trial_29_seed_3612.csv
Logged test evaluation CSV to MLflow artifacts path 'evaluation'
Skipping hybrid weight logging (LSTM-only model).
--- Finished Run: LSTM Optuna Trial 29 | Test MAE: 0.4439571797847748 ---

Optimization Finished!
Number of finished trials: 30
Best trial number: 16
  Value (MAE): 0.432193
  Best Parameters:
    lr: 0.0011276428963211847
    num_layers: 1
    batch_size: 32
    hidden_size: 80
    dropout: 0.01580166853138873

MLflow UI should show nested runs for each trial

### Example Workflow: Run Ensemble Training

In [ ]:
# 1. Define Configuration for Ensemble
ens_config = Config() # Start with defaults for LSTM
ens_config.random_seeds = [11, 22, 33] # Use a small set of seeds for testing
ens_config.num_epochs = 15 # Maybe fewer epochs for ensemble members
ens_config.ensemble_run_name = "hvac_ensemble_lstm_notebook_test"
ens_config.model_paths_filename = "ensemble_lstm_paths_notebook_test.json"

# 2. Run Ensemble Training
# This will create a parent MLflow run and nested runs for each seed
# Uses the simplified run_ensemble_training
print("--- Starting Ensemble Training (LSTM Only) ---")
parent_run_id, ensemble_info = run_ensemble_training(ens_config)
print("--- Ensemble Training Finished ---")

print(f"\nEnsemble training finished. Parent Run ID: {parent_run_id}")
print(f"Ensemble Info: {ensemble_info}")
# You can now use the 'ensemble_info' (run IDs or model paths)
# for ensemble prediction (logic not explicitly added here, but would involve loading multiple models
# from the paths/runs in ensemble_info and averaging their predictions).
# Prediction using an ensemble would require a new function similar to run_prediction_pipeline
# but modified to load multiple models and average their outputs.